# Explore the texts with BERTopic

Explore the preliminary TikTok corpus through captions, speech and on-screen text. The aim is to discover patterns, inspect examples and develop questions for later annotation. A topic is a model-generated grouping, not an established frame.

Run preprocessing and **04_build_tables** first. This notebook only reads the shared tables. It writes embeddings, tables and interactive HTML figures to your own home folder. Keep the same tables while running it; re-run from the import cell after rebuilding them.

We begin with **one document per post**, combining the available text sources. You can also select captions, transcriptions or OCR alone. Compare source coverage before interpreting differences between accounts. Missing speech/OCR can mean either no readable content or unfinished preprocessing.

Adapted from [Social Media Lab's BERTopic notebook](https://github.com/michaelachmann/social-media-lab/blob/main/notebooks/2024_11_18_BERTopic_V2.ipynb). See the [BERTopic documentation](https://maartengr.github.io/BERTopic/). GPL-3.0; the code license does not cover corpus data.

In [ ]:
%pip install -q --no-warn-script-location bertopic==0.17.4 sentence-transformers==6.3.0 transformers==5.23.0 umap-learn==0.6.0 hdbscan==0.8.44 plotly==7.1.0

# Packages installed in a running workspace may need the user site added.
import importlib, site
site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

## 1. Choose the material

Change `TEXT_MODE` to `combined`, `Caption`, `Transcription` or `OCR`. OCR always comes from **one** extraction strategy. `clean_then_raw` uses accepted cleaned text where available and otherwise the raw OCR; it never includes both versions of the same post. Set `OCR_SOURCE` to `clean` or `raw` to inspect either alone.

CPU is the default, so this notebook can run while preprocessing uses the GPU. Set `DEVICE = "cuda"` only in a GPU workspace when the shared GPU is available. On the seminar server, model weights are loaded from the administrator-prepared, read-only `/models/exploration` mount. No weights are copied into your home folder. Outside this setup, the first run downloads to your private Hugging Face cache. Embeddings are cached; changing clustering parameters does not repeat that work.

In [ ]:
import hashlib, json, os, re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, FileLink

TABLES = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived")) / "tables"
WORK = Path(os.environ.get("SEMINAR_EXPLORATION_HOME", str(Path.home() / "explorations"))) / "texts"
TEXT_MODE = os.environ.get("SEMINAR_TEXT_MODE", "combined")
OCR_STRATEGY = os.environ.get("SEMINAR_OCR_STRATEGY", "every_2s")
OCR_SOURCE = "clean_then_raw"  # clean_then_raw, clean, raw
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MODEL_REVISION = "main"  # Shared snapshot, or resolve an exact commit for a private download.
SHARED_MODEL_ROOT = Path(os.environ.get("SEMINAR_MODELS", "/models")) / "exploration"
USE_SHARED_MODEL = SHARED_MODEL_ROOT.is_dir()  # False only for deliberate private model downloads.
DEVICE = os.environ.get("SEMINAR_DEVICE", "cpu")
BATCH_SIZE = 16
SEED = 42
# Leave unset for the full selection; an evenly spaced subset is useful for a trial.
MAX_POSTS = int(os.environ["SEMINAR_MAX_POSTS"]) if os.environ.get("SEMINAR_MAX_POSTS") else None
MIN_CLUSTER_SIZE = 10
MIN_SAMPLES = 3

assert TEXT_MODE in {"combined", "Caption", "Transcription", "OCR"}
assert OCR_SOURCE in {"clean_then_raw", "clean", "raw"}
assert MAX_POSTS is None or MAX_POSTS >= 5
WORK.mkdir(parents=True, exist_ok=True)
CACHE = WORK / "cache"
CACHE.mkdir(exist_ok=True)
print("Results:", WORK)

In [ ]:
IDS = {"Post ID": str, "Text ID": str, "Image ID": str}
posts = pd.read_csv(TABLES / "posts.csv", dtype=IDS, keep_default_na=False)
texts = pd.read_csv(TABLES / "texts.csv", dtype=IDS, keep_default_na=False)
assert posts["Post ID"].is_unique and texts["Text ID"].is_unique
assert texts["Post ID"].isin(posts["Post ID"]).all()
assert not texts.duplicated(["Post ID", "Text Type", "Strategy"]).any(), "Rebuild the tables: repeated text sources."

print(f"{len(posts):,} posts; {len(texts):,} text rows")
coverage = texts.groupby(["Text Type", "Strategy"], dropna=False).agg(
    Texts=("Text ID", "size"), Posts=("Post ID", "nunique")
).reset_index()
display(coverage)
by_account = texts.groupby(["Username", "Text Type"]).size().unstack(fill_value=0)
display(posts.groupby("Username").size().rename("All posts").to_frame().join(by_account).fillna(0))

## 2. Build documents without counting OCR twice

The combined version retains source labels and all contributing `Text ID`s. Exactly repeated text within a post is included once. Repeated documents **across** posts are retained, because repetition may itself be interesting; their frequency is reported. Use the source tables to inspect what was combined.

In [ ]:
plain = texts[texts["Text Type"].isin(["Caption", "Transcription"])].copy()
raw = texts[(texts["Text Type"] == "OCR") & (texts["Strategy"] == OCR_STRATEGY)]
clean = texts[(texts["Text Type"] == "OCR Clean") & (texts["Strategy"] == OCR_STRATEGY)]
if OCR_SOURCE == "clean_then_raw":
    ocr = pd.concat([clean, raw[~raw["Post ID"].isin(clean["Post ID"])]])
elif OCR_SOURCE == "clean":
    ocr = clean.copy()
else:
    ocr = raw.copy()
selected = pd.concat([plain, ocr], ignore_index=True)
if TEXT_MODE == "OCR":
    selected = selected[selected["Text Type"].isin(["OCR", "OCR Clean"])]
elif TEXT_MODE != "combined":
    selected = selected[selected["Text Type"] == TEXT_MODE]

order = {"Caption": 0, "Transcription": 1, "OCR Clean": 2, "OCR": 2}
selected = selected.assign(_order=selected["Text Type"].map(order)).sort_values(["Post ID", "_order"])
rows = []
for post_id, group in selected.groupby("Post ID", sort=True):
    parts, seen, used_ids, sources = [], set(), [], []
    for _, row in group.iterrows():
        text = row["Text"].strip()
        if not text:
            continue
        used_ids.append(row["Text ID"])
        sources.append(row["Text Type"])
        normal = re.sub(r"\s+", " ", text)
        if normal in seen:
            continue
        seen.add(normal)
        parts.append(f"[{row['Text Type']}]\n{text}" if TEXT_MODE == "combined" else text)
    if parts:
        rows.append({"Post ID": post_id, "Document": "\n\n".join(parts),
                     "Text IDs": json.dumps(used_ids), "Sources": "; ".join(sources)})
documents = pd.DataFrame(rows, columns=["Post ID", "Document", "Text IDs", "Sources"])
# Require a word of at least two characters, excluding source labels themselves.
usable_posts = set(selected.loc[selected["Text"].str.contains(r"(?u)\b\w\w+\b", regex=True), "Post ID"])
documents = documents[documents["Post ID"].isin(usable_posts)]
documents = documents.merge(posts[["Post ID", "Username", "Created", "Plays"]], on="Post ID", validate="one_to_one")
if MAX_POSTS is not None and len(documents) > MAX_POSTS:
    documents = documents.iloc[np.linspace(0, len(documents)-1, MAX_POSTS, dtype=int)]
documents = documents.reset_index(drop=True)
assert len(documents) >= 5, "Select at least five posts with usable text."
print(f"Selected {len(documents):,}/{len(posts):,} posts; {documents['Document'].duplicated().sum():,} repeated documents.")
print("Sources used:", selected[selected["Post ID"].isin(documents["Post ID"])]["Text Type"].value_counts().to_dict())
display(documents.groupby("Username").size().rename("Selected posts").to_frame())
display(documents[["Post ID", "Sources", "Document"]].head())

## 3. Embed the documents once

We retain the original wording for embeddings. Long documents are split into token chunks that fit the model, embedded separately, and combined using a token-weighted mean. This prevents long transcripts being silently truncated, although the mean can still dilute individual themes.

The cache key includes the exact model revision, package version, IDs, text and chunking method. A different source selection or changed text gets a new cache. No corpus text is sent to a remote inference API; embedding runs in this workspace.

In [ ]:
import sentence_transformers
from huggingface_hub import HfApi
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

if USE_SHARED_MODEL:
    shared = (SHARED_MODEL_ROOT / MODEL.replace("/", "--") / "current").resolve(strict=True)
    manifest = json.loads((shared / "seminar-model.json").read_text())
    assert manifest["model"] == MODEL, "Shared model manifest does not match the selected model."
    revision = manifest["revision"]
    assert re.fullmatch(r"[0-9a-f]{40}", revision), "Expected an exact shared model revision."
    assert MODEL_REVISION in {"main", revision}, "Ask your instructor to prepare this revision, or use a private download."
    model_source = str(shared)
    load_options = {"local_files_only": True}
    print("Using read-only shared weights:", shared)
else:
    revision = MODEL_REVISION if re.fullmatch(r"[0-9a-f]{40}", MODEL_REVISION) else HfApi().model_info(MODEL, revision=MODEL_REVISION).sha
    model_source = MODEL
    load_options = {"revision": revision}
    print("No shared model selected: weights use your private Hugging Face cache.")
settings = {"model": MODEL, "revision": revision, "sentence_transformers": sentence_transformers.__version__,
            "method": "token_chunks_weighted_mean_v1", "mode": TEXT_MODE, "ocr_strategy": OCR_STRATEGY,
            "ocr_source": OCR_SOURCE}
fingerprint = hashlib.sha256(json.dumps({"settings": settings,
    "documents": documents[["Post ID", "Document"]].values.tolist()}, sort_keys=True, ensure_ascii=False).encode()).hexdigest()
run_id = fingerprint[:16]
cache_path = CACHE / f"{fingerprint}.npz"
ids = documents["Post ID"].to_numpy(dtype=str)

if cache_path.exists():
    with np.load(cache_path, allow_pickle=False) as saved:
        assert np.array_equal(saved["ids"], ids)
        embeddings = saved["embeddings"]
    print("Loaded cached embeddings:", cache_path.name)
else:
    encoder = SentenceTransformer(model_source, device=DEVICE, **load_options)
    tokenizer = encoder.tokenizer
    chunk_size = int(encoder.max_seq_length) - tokenizer.num_special_tokens_to_add(pair=False)
    assert chunk_size > 0
    vectors = []
    chunk_counts = []
    for text in tqdm(documents["Document"], desc="Embedding posts"):
        tokens = tokenizer.encode(text, add_special_tokens=False, truncation=False)
        pieces = [tokens[start:start+chunk_size] for start in range(0, len(tokens), chunk_size)]
        chunks = [tokenizer.decode(piece, skip_special_tokens=True) for piece in pieces]
        chunk_vectors = encoder.encode(chunks, batch_size=BATCH_SIZE, normalize_embeddings=False,
                                       convert_to_numpy=True, show_progress_bar=False)
        vectors.append(np.average(chunk_vectors, axis=0, weights=[len(piece) for piece in pieces]))
        chunk_counts.append(len(pieces))
    embeddings = np.asarray(vectors, dtype=np.float32)
    embeddings /= np.maximum(np.linalg.norm(embeddings, axis=1, keepdims=True), 1e-12)
    temporary = cache_path.with_suffix(".partial.npz")
    np.savez_compressed(temporary, ids=ids, embeddings=embeddings)
    temporary.replace(cache_path)
    print(f"Chunked {sum(n > 1 for n in chunk_counts)} long documents.")
    del encoder
    import gc
    gc.collect()
    if DEVICE.startswith("cuda"):
        import torch
        torch.cuda.empty_cache()
assert embeddings.ndim == 2 and len(embeddings) == len(documents) and np.isfinite(embeddings).all()
print("Embedding matrix:", embeddings.shape, "| exact model revision:", revision)

## 4. Find and inspect topics

Try a different `MIN_CLUSTER_SIZE`, then run from this section onwards. Smaller values can reveal smaller groups and can also produce fragile topics. `-1` means unassigned posts. Keep these visible instead of forcing every post into a topic.

The small editable stopword list below affects **topic descriptions**, not embeddings. Source headings are removed before word counting so they cannot become topic labels. Inspect the full post texts before naming a topic.

In [ ]:
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

STOP_WORDS = "aber als am an auch auf aus bei bin bis das dass dein deine dem den der des die doch du ein eine einem einen einer eines er es für hat ich im in ist mit nicht oder sich sie sind so und uns von war was wir zu zum zur the a an and are as at be for from in is it of on or that this to was with".split()
vectorizer = CountVectorizer(stop_words=STOP_WORDS, ngram_range=(1, 2))
# Do not let our own [Caption] etc. headings become topic keywords.
model_docs = [re.sub(r"(?m)^\[(Caption|Transcription|OCR Clean|OCR)\]\n", "", text) for text in documents["Document"]]
assert any(vectorizer.build_analyzer()(text) for text in model_docs), "No words remain; revise the stopword list."
n = len(documents)
assert 2 <= MIN_CLUSTER_SIZE <= n, "Choose a cluster size between 2 and the number of selected posts."
umap_model = UMAP(n_neighbors=min(15, n-1), n_components=min(5, n-2), metric="cosine",
                   random_state=SEED, n_jobs=1, init="random")
cluster_model = HDBSCAN(min_cluster_size=MIN_CLUSTER_SIZE, min_samples=min(MIN_SAMPLES, n-1),
                        metric="euclidean", prediction_data=True)
topic_model = BERTopic(embedding_model=None, umap_model=umap_model, hdbscan_model=cluster_model,
                      vectorizer_model=vectorizer, ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
                      calculate_probabilities=False, verbose=True)
topics, _ = topic_model.fit_transform(model_docs, embeddings=embeddings)
assignments = documents.assign(Topic=topics)
topic_info = topic_model.get_topic_info()
print(f"{sum(topic_info['Topic'] != -1)} topics; {(assignments['Topic'] == -1).sum()} unassigned posts")
display(topic_info[["Topic", "Count", "Name"]])
display(pd.crosstab(assignments["Username"], assignments["Topic"], normalize="index").round(2))

In [ ]:
# Select a topic, including -1, and read posts before interpreting the label.
TOPIC_TO_INSPECT = int(topic_info.sort_values("Count", ascending=False).iloc[0]["Topic"])
display(assignments.loc[assignments["Topic"] == TOPIC_TO_INSPECT,
                        ["Post ID", "Username", "Sources", "Document"]].head(10))
if TOPIC_TO_INSPECT != -1:
    print("Keywords:", topic_model.get_topic(TOPIC_TO_INSPECT))

## 5. Browse and save

The two-dimensional UMAP map is a browsing aid. Distances and visible islands are not validated thematic differences. Topic assignments above come from a separate reduction and clustering step.

Results use a fingerprint and parameter settings in their folder name, so changing the source selection or cluster size keeps earlier results. Embeddings are reusable; this notebook does not serialize a fitted BERTopic model.

In [ ]:
import html
from IPython.display import HTML
import plotly.express as px

run_settings = dict(settings, fingerprint=fingerprint, min_cluster_size=MIN_CLUSTER_SIZE,
                    min_samples=MIN_SAMPLES, seed=SEED, stop_words=STOP_WORDS)
parameter_id = hashlib.sha256(json.dumps(run_settings, sort_keys=True).encode()).hexdigest()[:10]
RUN = WORK / f"{run_id}-{parameter_id}"
RUN.mkdir(exist_ok=True)
coords = UMAP(n_neighbors=min(15, n-1), n_components=2, metric="cosine", random_state=SEED,
              n_jobs=1, init="random").fit_transform(embeddings)
assignments["Map X"], assignments["Map Y"] = coords[:, 0], coords[:, 1]
assignments["Topic label"] = assignments["Topic"].map(topic_info.set_index("Topic")["Name"])
plot_data = assignments.assign(
    Preview=assignments["Document"].map(lambda text: html.escape(text[:350]).replace("\n", "<br>")),
    Account=assignments["Username"].map(html.escape),
    Topic=assignments["Topic"].astype(str))
fig = px.scatter(plot_data, x="Map X", y="Map Y", color="Topic", hover_data=["Post ID", "Account", "Preview"],
                 title="Text similarity — colour shows topic assignment")
fig.write_html(RUN / "text-map.html", include_plotlyjs=True)
fig.show()
if (topic_info["Topic"] != -1).any():
    bars = topic_model.visualize_barchart(top_n_topics=min(12, sum(topic_info["Topic"] != -1)))
    bars.write_html(RUN / "topic-words.html", include_plotlyjs=True)
    bars.show()
assignments.to_csv(RUN / "post_topics.csv", index=False)
topic_info.to_csv(RUN / "topics.csv", index=False)
coverage.to_csv(RUN / "source_coverage.csv", index=False)
(RUN / "settings.json").write_text(json.dumps(run_settings, indent=2), encoding="utf-8")
print("Saved:", RUN)
# JupyterHub serves files relative to the home folder, not their absolute disk path.
from urllib.parse import quote
export = RUN / "text-map.html"
prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX")
if prefix and export.is_relative_to(Path.home()):
    url = prefix.rstrip("/") + "/files/" + quote(str(export.relative_to(Path.home())))
    display(HTML(f'<a href="{html.escape(url, quote=True)}" target="_blank">Open exported map</a>'))
else:
    display(FileLink(os.path.relpath(export, Path.cwd())))

## Questions for your first exploration

- Which topics reflect subject matter, and which reflect an account's repeated wording or genre?
- Read five posts from a topic: what connects them, and which example fits least well?
- What changes when you select transcriptions instead of captions? Compare which posts are available before attributing differences to modality.
- What can the unassigned posts tell you?
- Record a tentative observation and a counterexample. What would you need to annotate to investigate it?

**Before committing this notebook:** clear all outputs. Text examples and exported files contain corpus material and belong outside the public repository.